# Model Comparison & Feature Selection Demo

This notebook demonstrates two critical ML concepts using the geospatial site scoring pipeline:

1. **Task Comparison** — Regression (predict revenue) vs Lookalike Classification (identify top performers)
2. **Feature Selection Experiments** — How different feature subsets reveal bias, overfitting, and data leakage

All experiments use sklearn's **HistGradientBoosting** models (histogram-based gradient boosting, similar to XGBoost/LightGBM) with **permutation importance** for feature analysis.

---
## Section 1: Setup & Data Loading

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, precision_score, recall_score,
    log_loss, roc_curve, precision_recall_curve, confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Load training data
PROJECT_ROOT = Path('.').resolve()
train_pl = pl.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'site_training_data.parquet')
print(f'Loaded {train_pl.shape[0]:,} sites x {train_pl.shape[1]} columns')

In [ ]:
# Feature groups — exactly matching site_scoring/config.py
NUMERIC_FEATURES = [
    # Multi-horizon relative strength indicators
    'rs_NVIs_95_185', 'rs_Revenue_95_185',
    'rs_NVIs_185_370', 'rs_Revenue_185_370',
    'rs_NVIs_370_740', 'rs_Revenue_370_740',
    # Revenue metrics
    'log_total_revenue',
    # Geospatial distances
    'log_min_distance_to_nearest_site_mi', 'log_min_distance_to_interstate_mi',
    'log_min_distance_to_kroger_mi', 'log_min_distance_to_mcdonalds_mi',
    'log_min_distance_to_walmart_mi', 'log_min_distance_to_target_mi',
    # Demographics
    'log_avg_household_income', 'median_age', 'pct_female',
]

CATEGORICAL_FEATURES = [
    'network', 'program', 'experience_type', 'hardware_type',
    'retailer', 'brand_fuel', 'brand_c_store',
]

BOOLEAN_FEATURES = [
    'r_retail_car_wash_encoded', 'r_cpg_beverage_beer_oof_encoded',
    'r_cpg_beverage_beer_vide_encoded', 'r_cpg_beverage_wine_oof_encoded',
    'r_cpg_beverage_wine_video_encoded', 'r_finance_credit_cards_encoded',
    'r_cpg_cbd_hemp_ingestibles_non_thc_encoded',
    'r_cpg_non_food_beverage_cannabis_medical_encoded',
    'r_cpg_non_food_beverage_cannabis_recreational_encoded',
    'r_cpg_non_food_beverage_cbd_hemp_non_thc_encoded',
    'r_automotive_after_market_oil_encoded',
    'r_cpg_beverage_spirits_ooh_encoded', 'r_cpg_beverage_spirits_video_encoded',
    'r_cpg_non_food_beverage_e_cigarette_encoded',
    'r_entertainment_casinos_and_gambling_encoded',
    'r_government_political_encoded', 'r_automotive_electric_encoded',
    'r_recruitment_encoded', 'r_restaurants_cdr_encoded', 'r_restaurants_qsr_encoded',
    'r_retail_automotive_service_encoded', 'r_retail_grocery_encoded',
    'r_retail_grocerty_with_fuel_encoded',
    'c_emv_enabled_encoded', 'c_nfc_enabled_encoded', 'c_open_24_hours_encoded',
    'c_sells_beer_encoded', 'c_sells_diesel_fuel_encoded', 'c_sells_lottery_encoded',
    'c_vistar_programmatic_enabled_encoded', 'c_sells_wine_encoded',
    'schedule_site_encoded', 'sellable_site_encoded',
]

TARGET = 'avg_monthly_revenue'
LOOKALIKE_PERCENTILE = 90  # p90+ = top performers

# Filter to features that actually exist in the dataset
available_cols = set(train_pl.columns)
NUMERIC_FEATURES = [f for f in NUMERIC_FEATURES if f in available_cols]
CATEGORICAL_FEATURES = [f for f in CATEGORICAL_FEATURES if f in available_cols]
BOOLEAN_FEATURES = [f for f in BOOLEAN_FEATURES if f in available_cols]

print(f'Numeric:     {len(NUMERIC_FEATURES)} features')
print(f'Categorical: {len(CATEGORICAL_FEATURES)} features')
print(f'Boolean:     {len(BOOLEAN_FEATURES)} features')
print(f'Total:       {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES) + len(BOOLEAN_FEATURES)} features')

In [ ]:
# Prepare model-ready DataFrame
# Label-encode categoricals (HistGradientBoosting needs numeric input)
df = train_pl.to_pandas()

label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].fillna('__MISSING__').astype(str))
    label_encoders[col] = le

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BOOLEAN_FEATURES

# Fill NaN in numeric/boolean features
for col in NUMERIC_FEATURES + BOOLEAN_FEATURES:
    df[col] = df[col].fillna(0)

# Create regression target
y_reg = df[TARGET].values

# Create classification target (p90 threshold)
threshold = np.percentile(y_reg, LOOKALIKE_PERCENTILE)
y_cls = (y_reg >= threshold).astype(int)

# Feature matrix
X = df[ALL_FEATURES].values

# Train/test split (fixed seed for all experiments)
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')
print(f'\nRegression target: median=${np.median(y_reg):,.0f}, mean=${y_reg.mean():,.0f}')
print(f'Classification threshold (p{LOOKALIKE_PERCENTILE}): ${threshold:,.0f}')
print(f'  Positive class: {y_cls.sum():,} sites ({y_cls.mean()*100:.1f}%)')

# Shared model parameters
hgb_params = dict(
    max_iter=500, learning_rate=0.03, max_depth=6,
    random_state=42, verbose=0,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=50,
)
print(f'\nModel params: {hgb_params}')

---
## Section 2: Task Comparison — Regression vs Classification

Same features, same data, two fundamentally different questions:
- **Regression**: "How much revenue will this site generate?"
- **Classification**: "Is this site in the top 10% of performers?"

In [ ]:
# Train Regression Model
reg_model = HistGradientBoostingRegressor(loss='squared_error', **hgb_params)
reg_model.fit(X_train, y_reg_train)

y_reg_pred_train = reg_model.predict(X_train)
y_reg_pred_test = reg_model.predict(X_test)

reg_metrics = {
    'Train RMSE': np.sqrt(mean_squared_error(y_reg_train, y_reg_pred_train)),
    'Test RMSE': np.sqrt(mean_squared_error(y_reg_test, y_reg_pred_test)),
    'Train R2': r2_score(y_reg_train, y_reg_pred_train),
    'Test R2': r2_score(y_reg_test, y_reg_pred_test),
    'Train MAE': mean_absolute_error(y_reg_train, y_reg_pred_train),
    'Test MAE': mean_absolute_error(y_reg_test, y_reg_pred_test),
}

print('=== Regression Results ===')
for k, v in reg_metrics.items():
    fmt = f'${v:,.0f}' if 'MAE' in k or 'RMSE' in k else f'{v:.4f}'
    print(f'  {k:15s} {fmt}')
print(f'  Iterations used: {reg_model.n_iter_}')

In [ ]:
# Train Classification Model (Lookalike)
cls_model = HistGradientBoostingClassifier(loss='log_loss', **hgb_params)
cls_model.fit(X_train, y_cls_train)

y_cls_pred_test = cls_model.predict(X_test)
y_cls_prob_test = cls_model.predict_proba(X_test)[:, 1]
y_cls_prob_train = cls_model.predict_proba(X_train)[:, 1]

cls_metrics = {
    'Train AUC': roc_auc_score(y_cls_train, y_cls_prob_train),
    'Test AUC': roc_auc_score(y_cls_test, y_cls_prob_test),
    'Test F1': f1_score(y_cls_test, y_cls_pred_test),
    'Test Precision': precision_score(y_cls_test, y_cls_pred_test),
    'Test Recall': recall_score(y_cls_test, y_cls_pred_test),
    'Test Log Loss': log_loss(y_cls_test, y_cls_prob_test),
}

print('=== Classification Results ===')
for k, v in cls_metrics.items():
    print(f'  {k:17s} {v:.4f}')
print(f'  Iterations used: {cls_model.n_iter_}')

In [ ]:
# Side-by-side feature importance comparison using permutation importance
# Permutation importance measures how much test performance drops when a feature is shuffled —
# more reliable than gain-based importance because it accounts for feature interactions.
print('Computing permutation importance (this takes ~30s)...')

reg_perm = permutation_importance(reg_model, X_test, y_reg_test, n_repeats=10, random_state=42, n_jobs=-1)
reg_imp = pd.Series(reg_perm.importances_mean, index=ALL_FEATURES).sort_values(ascending=True)

cls_perm = permutation_importance(cls_model, X_test, y_cls_test, n_repeats=10, random_state=42, n_jobs=-1, scoring='roc_auc')
cls_imp = pd.Series(cls_perm.importances_mean, index=ALL_FEATURES).sort_values(ascending=True)

# Top 15 features from each
top_reg = reg_imp.tail(15)
top_cls = cls_imp.tail(15)
combined = sorted(set(top_reg.index) | set(top_cls.index))

fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(combined) * 0.35)))

# Regression importances
reg_vals = reg_imp.reindex(combined).sort_values()
reg_vals.plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Regression: Permutation Importance', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mean R2 decrease when shuffled')

# Classification importances
cls_vals = cls_imp.reindex(combined).sort_values()
cls_vals.plot(kind='barh', ax=axes[1], color='coral', alpha=0.8)
axes[1].set_title('Classification: Permutation Importance', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Mean AUC decrease when shuffled')

plt.tight_layout()
plt.show()

# Rank correlation between the two importance orderings
from scipy.stats import spearmanr
rho, pval = spearmanr(reg_imp.values, cls_imp.reindex(reg_imp.index).values)
print(f'\nRank correlation (Spearman) between regression and classification importance: rho = {rho:.3f} (p = {pval:.2e})')
print('-> High correlation means both tasks care about similar features.')
print('-> Low correlation means the boundary (classification) uses different signals than the curve (regression).')

In [ ]:
# Prediction analysis: Regression residuals + Classification curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted (Regression)
ax = axes[0, 0]
ax.scatter(y_reg_test, y_reg_pred_test, alpha=0.1, s=3, color='steelblue')
lims = [min(y_reg_test.min(), y_reg_pred_test.min()), np.percentile(y_reg_test, 99)]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Revenue ($)')
ax.set_ylabel('Predicted Revenue ($)')
ax.set_title('Regression: Actual vs Predicted')
ax.legend()

# 2. Residuals distribution
ax = axes[0, 1]
residuals = y_reg_test - y_reg_pred_test
ax.hist(residuals, bins=80, color='steelblue', alpha=0.7, edgecolor='none')
ax.axvline(0, color='red', linestyle='--')
ax.set_title(f'Regression: Residuals (mean=${residuals.mean():,.0f})')
ax.set_xlabel('Residual (actual - predicted)')

# 3. ROC Curve (Classification)
ax = axes[1, 0]
fpr, tpr, _ = roc_curve(y_cls_test, y_cls_prob_test)
ax.plot(fpr, tpr, color='coral', linewidth=2, label=f'AUC = {cls_metrics["Test AUC"]:.3f}')
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Classification: ROC Curve')
ax.legend()

# 4. Precision-Recall Curve
ax = axes[1, 1]
prec, rec, _ = precision_recall_curve(y_cls_test, y_cls_prob_test)
ax.plot(rec, prec, color='coral', linewidth=2)
baseline = y_cls_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Classification: Precision-Recall')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Prediction agreement/disagreement analysis
# Where do regression and classification agree vs disagree?
reg_high = y_reg_pred_test >= threshold  # Regression predicts high revenue
cls_positive = y_cls_pred_test == 1       # Classification predicts top performer

agreement_matrix = pd.DataFrame({
    'Classification: Top': [
        (reg_high & cls_positive).sum(),
        (~reg_high & cls_positive).sum(),
    ],
    'Classification: Not Top': [
        (reg_high & ~cls_positive).sum(),
        (~reg_high & ~cls_positive).sum(),
    ],
}, index=['Regression: High', 'Regression: Low'])

print('=== Prediction Agreement Matrix ===')
print(agreement_matrix.to_string())
total = len(y_reg_test)
agree = (reg_high & cls_positive).sum() + (~reg_high & ~cls_positive).sum()
print(f'\nAgreement rate: {agree}/{total} = {agree/total*100:.1f}%')
print(f'Disagreement:   {total-agree}/{total} = {(total-agree)/total*100:.1f}%')
print('\n→ Disagreements happen because regression fits the entire revenue curve')
print('  while classification only models the boundary at the p90 threshold.')

---
## Section 3: Feature Subset Definitions

We define 6 feature subsets to test how feature selection affects model quality.
Each subset tests a different hypothesis about bias, variance, and data leakage.

In [ ]:
# Define 6 feature subsets
available_in_df = set(df.columns)

FEATURE_SUBSETS = {
    'All Features': ALL_FEATURES,

    'Revenue-Leaky': [
        # DANGER: These are derived from / correlated with the target
        f for f in [
            'total_revenue', 'log_total_revenue',
            'total_monthly_impressions', 'total_monthly_nvis',
            'total_monthly_revenue_per_screen',
            'avg_monthly_monthly_impressions', 'avg_monthly_monthly_nvis',
            'active_months',
        ] if f in available_in_df
    ],

    'Demographics Only': [
        f for f in [
            'log_avg_household_income', 'median_age',
            'pct_female', 'pct_african_american', 'pct_asian', 'pct_hispanic',
        ] if f in available_in_df
    ],

    'Geospatial Only': [
        f for f in [
            'log_min_distance_to_nearest_site_mi', 'log_min_distance_to_interstate_mi',
            'log_min_distance_to_kroger_mi', 'log_min_distance_to_mcdonalds_mi',
            'log_min_distance_to_walmart_mi', 'log_min_distance_to_target_mi',
            'dma_rank',
        ] if f in available_in_df
    ],

    'Momentum Only': [
        f for f in [
            'rs_Impressions_95_185', 'rs_NVIs_95_185', 'rs_Revenue_95_185', 'rs_RevenuePerScreen_95_185',
            'rs_Impressions_185_370', 'rs_NVIs_185_370', 'rs_Revenue_185_370', 'rs_RevenuePerScreen_185_370',
            'rs_Impressions_370_740', 'rs_NVIs_370_740', 'rs_Revenue_370_740', 'rs_RevenuePerScreen_370_740',
        ] if f in available_in_df
    ],

    'Boolean Flags Only': BOOLEAN_FEATURES,
}

# Summary table
print(f'{"Subset":<22s} {"Features":>8s}  Hypothesis')
print('-' * 80)
hypotheses = {
    'All Features': 'Baseline — best generalization expected',
    'Revenue-Leaky': 'DATA LEAKAGE — target-derived features inflate metrics',
    'Demographics Only': 'Underfitting — too few features, but stable',
    'Geospatial Only': 'Location signal only — moderate performance',
    'Momentum Only': 'Temporal trends — captures recent performance changes',
    'Boolean Flags Only': 'High dimensionality, low variance — overfitting risk',
}
for name, features in FEATURE_SUBSETS.items():
    print(f'{name:<22s} {len(features):>8d}  {hypotheses[name]}')

---
## Section 4: Feature Subset Experiments — Regression

Train the same HistGradientBoosting regressor on each feature subset and compare train vs test metrics.

In [ ]:
# Train regression model on each feature subset
reg_results = {}

for name, features in FEATURE_SUBSETS.items():
    if not features:
        print(f'  Skipping {name} -- no features available')
        continue

    # Build matrices directly from df (some subset features may not be in ALL_FEATURES)
    X_sub = df[features].fillna(0).values
    X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y_reg, test_size=0.2, random_state=42)

    model = HistGradientBoostingRegressor(loss='squared_error', **hgb_params)
    model.fit(X_tr, y_tr)

    train_pred = model.predict(X_tr)
    test_pred = model.predict(X_te)

    reg_results[name] = {
        'n_features': len(features),
        'train_rmse': np.sqrt(mean_squared_error(y_tr, train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_te, test_pred)),
        'train_r2': r2_score(y_tr, train_pred),
        'test_r2': r2_score(y_te, test_pred),
        'train_mae': mean_absolute_error(y_tr, train_pred),
        'test_mae': mean_absolute_error(y_te, test_pred),
        'model': model,
        'features': features,
        'X_test': X_te,
        'y_test': y_te,
    }
    gap = reg_results[name]['train_r2'] - reg_results[name]['test_r2']
    print(f'{name:22s} | R2 train={reg_results[name]["train_r2"]:.4f}  test={reg_results[name]["test_r2"]:.4f}  gap={gap:.4f}')

print('\n-> Large train-test gap = overfitting. Suspiciously high metrics = data leakage.')

In [ ]:
# Results table with color coding
results_df = pd.DataFrame({
    name: {
        'Features': r['n_features'],
        'Train R²': r['train_r2'],
        'Test R²': r['test_r2'],
        'R² Gap': r['train_r2'] - r['test_r2'],
        'Train RMSE': r['train_rmse'],
        'Test RMSE': r['test_rmse'],
    } for name, r in reg_results.items()
}).T

results_df['Features'] = results_df['Features'].astype(int)

# Style: highlight leakage and overfitting
def highlight_issues(row):
    styles = [''] * len(row)
    if row['R² Gap'] > 0.1:  # Large gap = overfitting
        styles[3] = 'background-color: #ffcccc'  # red
    if row['Test R²'] > 0.95:  # Suspiciously high = possible leakage
        styles[2] = 'background-color: #ffffcc'  # yellow
    return styles

results_df.style.apply(highlight_issues, axis=1).format({
    'Train R²': '{:.4f}', 'Test R²': '{:.4f}', 'R² Gap': '{:.4f}',
    'Train RMSE': '${:,.0f}', 'Test RMSE': '${:,.0f}',
})

In [ ]:
# Train vs Test gap visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

subset_names = list(reg_results.keys())
x = np.arange(len(subset_names))
width = 0.35

# R² comparison
ax = axes[0]
train_r2 = [reg_results[n]['train_r2'] for n in subset_names]
test_r2 = [reg_results[n]['test_r2'] for n in subset_names]
bars1 = ax.bar(x - width/2, train_r2, width, label='Train R²', color='steelblue', alpha=0.5)
bars2 = ax.bar(x + width/2, test_r2, width, label='Test R²', color='steelblue', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(subset_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('R²')
ax.set_title('Regression R²: Train vs Test', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)

# RMSE comparison
ax = axes[1]
train_rmse = [reg_results[n]['train_rmse'] for n in subset_names]
test_rmse = [reg_results[n]['test_rmse'] for n in subset_names]
ax.bar(x - width/2, train_rmse, width, label='Train RMSE', color='coral', alpha=0.5)
ax.bar(x + width/2, test_rmse, width, label='Test RMSE', color='coral', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(subset_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('RMSE ($)')
ax.set_title('Regression RMSE: Train vs Test', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance per subset (top 10 for each, using permutation importance)
print('Computing permutation importance per subset...')
n_subsets = len(reg_results)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, r) in enumerate(reg_results.items()):
    if i >= len(axes):
        break
    ax = axes[i]
    perm = permutation_importance(r['model'], r['X_test'], r['y_test'],
                                  n_repeats=5, random_state=42, n_jobs=-1)
    imp = pd.Series(perm.importances_mean, index=r['features'])
    top10 = imp.nlargest(10).sort_values()
    top10.plot(kind='barh', ax=ax, color='teal', alpha=0.8)
    ax.set_title(f'{name}\n(Test R2 = {r["test_r2"]:.3f})', fontsize=10)
    ax.set_xlabel('Permutation Importance')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top 10 Feature Importances by Subset (Regression)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Section 5: Feature Subset Experiments — Classification

Same 6 subsets, but now predicting "top performer" (p90+) instead of continuous revenue.

In [ ]:
# Train classification model on each feature subset
cls_results = {}

for name, features in FEATURE_SUBSETS.items():
    if not features:
        continue

    X_sub = df[features].fillna(0).values
    X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y_cls, test_size=0.2, random_state=42)

    model = HistGradientBoostingClassifier(loss='log_loss', **hgb_params)
    model.fit(X_tr, y_tr)

    prob_train = model.predict_proba(X_tr)[:, 1]
    prob_test = model.predict_proba(X_te)[:, 1]
    pred_test = model.predict(X_te)

    cls_results[name] = {
        'n_features': len(features),
        'train_auc': roc_auc_score(y_tr, prob_train),
        'test_auc': roc_auc_score(y_te, prob_test),
        'auc_gap': roc_auc_score(y_tr, prob_train) - roc_auc_score(y_te, prob_test),
        'test_f1': f1_score(y_te, pred_test),
        'test_precision': precision_score(y_te, pred_test),
        'test_recall': recall_score(y_te, pred_test),
        'prob_test': prob_test,
        'y_test': y_te,
    }
    gap = cls_results[name]['auc_gap']
    print(f'{name:22s} | AUC train={cls_results[name]["train_auc"]:.4f}  test={cls_results[name]["test_auc"]:.4f}  gap={gap:.4f}  F1={cls_results[name]["test_f1"]:.3f}')

In [ ]:
# ROC curves for all subsets overlaid
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.Set2(np.linspace(0, 1, len(cls_results)))

# ROC curves
ax = axes[0]
for (name, r), color in zip(cls_results.items(), colors):
    fpr, tpr, _ = roc_curve(r['y_test'], r['prob_test'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} (AUC={r["test_auc"]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Classification ROC by Feature Subset', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')

# Precision-Recall curves
ax = axes[1]
for (name, r), color in zip(cls_results.items(), colors):
    prec, rec, _ = precision_recall_curve(r['y_test'], r['prob_test'])
    ax.plot(rec, prec, color=color, linewidth=2, label=name)
baseline = y_cls.mean()
ax.axhline(baseline, color='gray', linestyle='--', linewidth=0.8, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall by Feature Subset', fontweight='bold')
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Classification results table
cls_df = pd.DataFrame({
    name: {
        'Features': r['n_features'],
        'Train AUC': r['train_auc'],
        'Test AUC': r['test_auc'],
        'AUC Gap': r['auc_gap'],
        'F1': r['test_f1'],
        'Precision': r['test_precision'],
        'Recall': r['test_recall'],
    } for name, r in cls_results.items()
}).T

cls_df['Features'] = cls_df['Features'].astype(int)

cls_df.style.format({
    'Train AUC': '{:.4f}', 'Test AUC': '{:.4f}', 'AUC Gap': '{:.4f}',
    'F1': '{:.3f}', 'Precision': '{:.3f}', 'Recall': '{:.3f}',
})

---
## Section 6: Overfitting Deep Dive

### Learning Curves
How does performance change as we vary training data size? Overfitting models show high train / low test that doesn't converge.

In [ ]:
# Learning curves for 3 key subsets
subsets_for_curves = ['All Features', 'Demographics Only', 'Boolean Flags Only']
data_fractions = [0.1, 0.25, 0.5, 0.75, 1.0]

learning_curves = {}

# Use early_stopping=False for learning curves to avoid issues with small fractions
lc_params = dict(
    max_iter=200, learning_rate=0.03, max_depth=6,
    random_state=42, verbose=0, early_stopping=False,
)

for name in subsets_for_curves:
    if name not in FEATURE_SUBSETS:
        continue
    features = FEATURE_SUBSETS[name]
    X_sub = df[features].fillna(0).values
    X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y_reg, test_size=0.2, random_state=42)

    curves = {'fractions': [], 'train_r2': [], 'test_r2': []}

    for frac in data_fractions:
        n_samples = int(len(X_tr) * frac)
        X_tr_sub = X_tr[:n_samples]
        y_tr_sub = y_tr[:n_samples]

        model = HistGradientBoostingRegressor(loss='squared_error', **lc_params)
        model.fit(X_tr_sub, y_tr_sub)

        curves['fractions'].append(frac)
        curves['train_r2'].append(r2_score(y_tr_sub, model.predict(X_tr_sub)))
        curves['test_r2'].append(r2_score(y_te, model.predict(X_te)))

    learning_curves[name] = curves
    print(f'{name}: done')

# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'All Features': 'steelblue', 'Demographics Only': 'seagreen', 'Boolean Flags Only': 'coral'}

for i, name in enumerate(subsets_for_curves):
    if name not in learning_curves:
        continue
    c = learning_curves[name]
    ax = axes[i]
    ax.plot(c['fractions'], c['train_r2'], 'o-', color=colors[name], alpha=0.5, label='Train R2')
    ax.plot(c['fractions'], c['test_r2'], 's-', color=colors[name], linewidth=2, label='Test R2')
    ax.fill_between(c['fractions'], c['train_r2'], c['test_r2'], alpha=0.15, color=colors[name])
    ax.set_xlabel('Training Data Fraction')
    ax.set_ylabel('R2')
    ax.set_title(f'{name}\n({len(FEATURE_SUBSETS[name])} features)', fontweight='bold')
    ax.legend()
    ax.set_ylim(-0.1, 1.05)

plt.suptitle('Learning Curves: Train vs Test R2 by Data Size', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('-> Boolean Flags: train stays high, test stays low = classic overfitting.')
print('-> Demographics Only: both low but close together = high bias (underfitting).')
print('-> All Features: test improves with more data and gap closes = good generalization.')

In [ ]:
# Cross-validation stability
cv_results = {}
print('Running 5-fold cross-validation per subset...\n')

cv_model_params = dict(
    max_iter=200, learning_rate=0.03, max_depth=6,
    random_state=42, verbose=0, early_stopping=True,
    validation_fraction=0.15, n_iter_no_change=20,
)

for name, features in FEATURE_SUBSETS.items():
    if not features:
        continue
    X_sub = df[features].fillna(0).values

    model = HistGradientBoostingRegressor(loss='squared_error', **cv_model_params)
    scores = cross_val_score(model, X_sub, y_reg, cv=5, scoring='r2')
    cv_results[name] = scores
    print(f'{name:22s} | R2 = {scores.mean():.4f} +/- {scores.std():.4f}  (range: {scores.min():.4f} - {scores.max():.4f})')

# Box plot
fig, ax = plt.subplots(figsize=(12, 5))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(ax=ax, vert=True, patch_artist=True,
              boxprops=dict(facecolor='steelblue', alpha=0.3),
              medianprops=dict(color='red', linewidth=2))
ax.set_ylabel('R2 (5-fold CV)')
ax.set_title('Cross-Validation Stability: R2 Distribution per Feature Subset', fontweight='bold')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

print('\n-> High variance across folds = unstable model (overfitting to specific data splits).')
print('-> Consistently high with low variance = either good model or data leakage.')

In [ ]:
# Bias-Variance visualization
fig, ax = plt.subplots(figsize=(10, 7))

for name, scores in cv_results.items():
    # Bias ≈ 1 - mean(R²)  (how far from perfect)
    # Variance ≈ std(R²)   (how unstable across folds)
    bias = 1 - scores.mean()
    variance = scores.std()

    ax.scatter(bias, variance, s=200, zorder=5)
    ax.annotate(f'{name}\n({len(FEATURE_SUBSETS[name])}f)',
                (bias, variance), fontsize=9, ha='center', va='bottom',
                xytext=(0, 10), textcoords='offset points')

ax.set_xlabel('Bias (1 - mean R²) →  Higher = Underfitting', fontsize=11)
ax.set_ylabel('Variance (std R²) →  Higher = Overfitting', fontsize=11)
ax.set_title('Bias-Variance Tradeoff by Feature Subset', fontsize=13, fontweight='bold')

# Add quadrant labels
xlim = ax.get_xlim()
ylim = ax.get_ylim()
mid_x = (xlim[0] + xlim[1]) / 2
mid_y = (ylim[0] + ylim[1]) / 2
ax.text(xlim[0] + 0.02, ylim[1] - 0.002, 'Good fit but unstable', fontsize=8, color='gray', style='italic')
ax.text(xlim[1] - 0.15, ylim[1] - 0.002, 'Bad: high bias + variance', fontsize=8, color='gray', style='italic')
ax.text(xlim[0] + 0.02, ylim[0] + 0.001, 'IDEAL: low bias, low variance', fontsize=8, color='green', style='italic', fontweight='bold')
ax.text(xlim[1] - 0.15, ylim[0] + 0.001, 'Underfitting (stable but wrong)', fontsize=8, color='gray', style='italic')

ax.axvline(mid_x, color='lightgray', linestyle=':', linewidth=0.8)
ax.axhline(mid_y, color='lightgray', linestyle=':', linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Data Leakage Detection
# Demonstrate why 'Revenue-Leaky' subset is cheating

leaky_cols = ['total_revenue', 'avg_monthly_revenue', 'active_months']
leaky_available = [c for c in leaky_cols if c in df.columns]

if len(leaky_available) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Show perfect correlation: avg_monthly_revenue ≈ total_revenue / active_months
    ax = axes[0]
    if 'total_revenue' in df.columns and 'active_months' in df.columns:
        derived = df['total_revenue'] / df['active_months'].clip(lower=1)
        actual = df['avg_monthly_revenue']
        ax.scatter(actual, derived, alpha=0.05, s=1, color='red')
        ax.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'k--')
        corr = np.corrcoef(actual.fillna(0), derived.fillna(0))[0, 1]
        ax.set_title(f'avg_monthly_revenue vs total_revenue/active_months\n(correlation = {corr:.6f})')
        ax.set_xlabel('avg_monthly_revenue (TARGET)')
        ax.set_ylabel('total_revenue / active_months (FEATURE)')

    # Show log_total_revenue vs target correlation
    ax = axes[1]
    if 'log_total_revenue' in df.columns:
        ax.scatter(df['avg_monthly_revenue'], df['log_total_revenue'], alpha=0.05, s=1, color='orange')
        corr = np.corrcoef(
            df['avg_monthly_revenue'].fillna(0),
            df['log_total_revenue'].fillna(0)
        )[0, 1]
        ax.set_title(f'Target vs log_total_revenue\n(correlation = {corr:.4f})')
        ax.set_xlabel('avg_monthly_revenue (TARGET)')
        ax.set_ylabel('log_total_revenue (FEATURE)')

    plt.suptitle('DATA LEAKAGE: These features are derived from the target!',
                 fontsize=13, fontweight='bold', color='red', y=1.02)
    plt.tight_layout()
    plt.show()

    print('LESSON: Always check if your features are derived from or correlated with the target.')
    print('total_revenue and active_months together perfectly reconstruct avg_monthly_revenue.')
    print('A model using these will score perfectly — but learn nothing about site characteristics.')
else:
    print('Leaky columns not available for visualization.')

---
## Section 7: Key Takeaways

In [ ]:
# Master summary table: all experiments
summary_rows = []
for name in FEATURE_SUBSETS:
    row = {'Subset': name, 'Features': len(FEATURE_SUBSETS[name])}
    if name in reg_results:
        row['Reg Train R²'] = reg_results[name]['train_r2']
        row['Reg Test R²'] = reg_results[name]['test_r2']
        row['Reg Gap'] = reg_results[name]['train_r2'] - reg_results[name]['test_r2']
    if name in cls_results:
        row['Cls Train AUC'] = cls_results[name]['train_auc']
        row['Cls Test AUC'] = cls_results[name]['test_auc']
        row['Cls Gap'] = cls_results[name]['auc_gap']
    if name in cv_results:
        row['CV R² (mean±std)'] = f"{cv_results[name].mean():.3f}±{cv_results[name].std():.3f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Subset')

# Sort by test R²
if 'Reg Test R²' in summary_df.columns:
    summary_df = summary_df.sort_values('Reg Test R²', ascending=False)

print('=== MASTER COMPARISON TABLE ===')
print('(Sorted by Regression Test R²)\n')
print(summary_df.to_string(float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else x))

# Flag issues
print('\n--- Flags ---')
for name in FEATURE_SUBSETS:
    if name in reg_results:
        gap = reg_results[name]['train_r2'] - reg_results[name]['test_r2']
        test_r2 = reg_results[name]['test_r2']
        if test_r2 > 0.95:
            print(f'  ⚠️  {name}: Test R² = {test_r2:.4f} — suspiciously high (possible data leakage)')
        elif gap > 0.1:
            print(f'  🔴 {name}: Train-Test gap = {gap:.4f} — overfitting')
        elif gap < 0.02 and test_r2 > 0.3:
            print(f'  ✅ {name}: Good generalization (gap = {gap:.4f}, Test R² = {test_r2:.4f})')

### Conclusions

**Regression vs Classification:**
- Regression fits the full revenue curve — better for *predicting* revenue
- Classification finds the decision boundary — better for *identifying* top sites
- Feature importance rankings differ because each task optimizes a different objective

**Feature Selection:**
- **More features ≠ better model** — Boolean flags (40 features) overfit with little signal
- **Data leakage is the #1 risk** — `total_revenue` perfectly reconstructs the target
- **Demographics alone underfit** — too few features to capture site complexity
- **Geospatial + Momentum features** provide the most generalizable, non-leaky signal
- **Cross-validation reveals instability** that a single train/test split hides

**Production Recommendation:**  
Use the curated "All Features" set from `config.py` which excludes leaky features and low-variance flags. Monitor the train-test gap as a health metric.

---
## Section 8: Bonus — Incremental Feature Addition

In [ ]:
# Greedy forward feature selection
# Start with best single feature, add one at a time
print('Running greedy forward feature selection (this may take a few minutes)...\n')

remaining = list(ALL_FEATURES)
selected = []
history = []

# Limit to first 20 additions for time
max_features = min(20, len(remaining))

fwd_params = dict(
    max_iter=100, learning_rate=0.05, max_depth=4,
    random_state=42, verbose=0, early_stopping=False,
)

for step in range(max_features):
    best_score = -np.inf
    best_feature = None

    for feat in remaining:
        trial = selected + [feat]
        X_sub = df[trial].fillna(0).values
        X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y_reg, test_size=0.2, random_state=42)

        model = HistGradientBoostingRegressor(loss='squared_error', **fwd_params)
        model.fit(X_tr, y_tr)
        score = r2_score(y_te, model.predict(X_te))

        if score > best_score:
            best_score = score
            best_feature = feat

    selected.append(best_feature)
    remaining.remove(best_feature)
    history.append({'n_features': len(selected), 'feature': best_feature, 'test_r2': best_score})
    print(f'  Step {step+1:2d}: +{best_feature:45s} -> R2 = {best_score:.4f}')

# Plot diminishing returns
fig, ax = plt.subplots(figsize=(12, 5))
n_feats = [h['n_features'] for h in history]
scores = [h['test_r2'] for h in history]
ax.plot(n_feats, scores, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.set_xlabel('Number of Features')
ax.set_ylabel('Test R2')
ax.set_title('Greedy Forward Selection: Diminishing Returns', fontweight='bold')

# Annotate the first few features
for h in history[:5]:
    ax.annotate(h['feature'].replace('log_min_distance_to_', 'dist_').replace('_encoded', ''),
                (h['n_features'], h['test_r2']),
                fontsize=7, rotation=30, ha='left', va='bottom',
                xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

print(f'\n-> Most gain in first 5 features. After ~15, additional features add minimal value.')

In [ ]:
# Feature importance stability across random seeds
print('Training 5 models with different random seeds and computing permutation importance...\n')

importance_runs = []
for seed in [42, 123, 456, 789, 1024]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[ALL_FEATURES].fillna(0).values, y_reg, test_size=0.2, random_state=seed
    )
    model = HistGradientBoostingRegressor(
        loss='squared_error', max_iter=300, learning_rate=0.03, max_depth=6,
        random_state=seed, verbose=0, early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=30,
    )
    model.fit(X_tr, y_tr)

    # Permutation importance (5 repeats for speed)
    perm = permutation_importance(model, X_te, y_te, n_repeats=5,
                                  random_state=seed, n_jobs=-1)
    imp = pd.Series(perm.importances_mean, index=ALL_FEATURES)
    importance_runs.append(imp)
    print(f'  Seed {seed}: R2 = {r2_score(y_te, model.predict(X_te)):.4f}')

imp_df = pd.DataFrame(importance_runs)
imp_mean = imp_df.mean().sort_values(ascending=False)
imp_std = imp_df.std()

# Top 20 features by mean importance with stability bars
top20 = imp_mean.head(20)
top20_std = imp_std.reindex(top20.index)

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top20))
ax.barh(y_pos, top20.values[::-1], xerr=top20_std.values[::-1],
        color='teal', alpha=0.7, capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(top20.index[::-1], fontsize=9)
ax.set_xlabel('Permutation Importance (mean +/- std across 5 seeds)')
ax.set_title('Feature Importance Stability (Top 20)', fontweight='bold')
plt.tight_layout()
plt.show()

# Coefficient of variation (CV) -- low = stable
cv = (imp_std / imp_mean.clip(lower=1e-6)).sort_values()
print('\nMost stable features (lowest coefficient of variation):')
for feat in cv.head(10).index:
    print(f'  {feat:45s} CV = {cv[feat]:.3f}  (mean imp = {imp_mean[feat]:.4f})')
print('\n-> Stable features are more trustworthy for production deployment.')